# 📊 Imbalance Labs — EDA Starter Kit

This notebook walks you through loading, exploring, and visualizing the Imbalance Labs orderbook dataset.

**Dataset specs:**
- 24 crypto instruments (BTC, ETH, SOL, etc.)
- 5-minute aggregated bars, 12+ months history
- 47 columns per row: OHLC + 10-level depth (bid/ask volumes + distances)

---

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

ACCENT = '#00FF88'

# ── Load a single instrument ──
# Replace with your file path
df = pd.read_csv('BTC_5m_depth10_derived.csv.gz')
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])
df = df.set_index('timestamp_utc').sort_index()

print(f'Instrument: {df["instrument_symbol"].iloc[0]}')
print(f'Rows: {len(df):,}')
print(f'Date range: {df.index.min()} → {df.index.max()}')
print(f'Columns: {len(df.columns)}')
df.head()

## 2. Price Overview

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(df.index, df['close_price'], color=ACCENT, linewidth=0.7, alpha=0.9)
ax1.fill_between(df.index, df['low_price'], df['high_price'], alpha=0.1, color=ACCENT)
ax1.set_ylabel('Price (USDT)')
ax1.set_title(f'{df["instrument_symbol"].iloc[0]} — Close Price', fontweight='bold')
ax1.grid(alpha=0.15)

ax2.bar(df.index, df['interval_traded_volume'], width=0.003, color=ACCENT, alpha=0.5)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')
ax2.grid(alpha=0.15)

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()

## 3. Orderbook Depth Profile

Visualize how liquidity is distributed across the 10 depth levels.

In [ ]:
# Average depth profile across entire dataset
bid_cols = [f'bid_volume_level_{i}' for i in range(1, 11)]
ask_cols = [f'ask_volume_level_{i}' for i in range(1, 11)]

avg_bid = df[bid_cols].mean().values
avg_ask = df[ask_cols].mean().values

levels = np.arange(1, 11)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(levels - 0.2, avg_bid, height=0.35, color='#00FF88', alpha=0.8, label='Bid (Buy Wall)')
ax.barh(levels + 0.2, avg_ask, height=0.35, color='#FF4444', alpha=0.8, label='Ask (Sell Wall)')
ax.set_ylabel('Depth Level')
ax.set_xlabel('Average Cumulative Volume')
ax.set_title('Orderbook Depth Profile — Average Liquidity by Level', fontweight='bold')
ax.set_yticks(levels)
ax.legend()
ax.grid(alpha=0.15, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Bid-Ask Imbalance

The **imbalance ratio** measures the relative pressure between buyers and sellers at each level.

$$\text{Imbalance}_k = \frac{\text{Bid}_k - \text{Ask}_k}{\text{Bid}_k + \text{Ask}_k}$$

A value close to **+1** means heavy buy-side pressure; **−1** means sell-side dominance.

In [ ]:
# Compute imbalance for level 1 (tightest)
df['imbalance_L1'] = (
    (df['bid_volume_level_1'] - df['ask_volume_level_1']) /
    (df['bid_volume_level_1'] + df['ask_volume_level_1'])
)

# Rolling smoothed version
df['imbalance_L1_smooth'] = df['imbalance_L1'].rolling(60).mean()  # 5h rolling avg

# ── Plot ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(df.index, df['close_price'], color=ACCENT, linewidth=0.6)
ax1.set_ylabel('Price')
ax1.set_title('Price vs Level-1 Bid-Ask Imbalance', fontweight='bold')
ax1.grid(alpha=0.15)

ax2.fill_between(df.index, df['imbalance_L1_smooth'],
                 where=df['imbalance_L1_smooth'] > 0, color='#00FF88', alpha=0.5, label='Buy pressure')
ax2.fill_between(df.index, df['imbalance_L1_smooth'],
                 where=df['imbalance_L1_smooth'] < 0, color='#FF4444', alpha=0.5, label='Sell pressure')
ax2.axhline(0, color='white', linewidth=0.5, alpha=0.3)
ax2.set_ylabel('Imbalance (L1)')
ax2.set_ylim(-0.5, 0.5)
ax2.legend(loc='upper right')
ax2.grid(alpha=0.15)

plt.tight_layout()
plt.show()

## 5. Depth Distance Heatmap

How far is the liquidity from mid-price at each level? Tighter spread = more liquid market.

In [ ]:
bid_dist_cols = [f'bid_distance_level_{i}' for i in range(1, 11)]
ask_dist_cols = [f'ask_distance_level_{i}' for i in range(1, 11)]

# Resample to daily for cleaner heatmap
daily_bid_dist = df[bid_dist_cols].resample('1D').mean()
daily_ask_dist = df[ask_dist_cols].resample('1D').mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

im1 = ax1.imshow(daily_bid_dist.T, aspect='auto', cmap='YlGn', interpolation='nearest')
ax1.set_title('Bid Distance from Mid-Price (bps)', fontweight='bold')
ax1.set_ylabel('Depth Level')
ax1.set_yticks(range(10))
ax1.set_yticklabels([f'L{i}' for i in range(1, 11)])
ax1.set_xlabel('Day')
plt.colorbar(im1, ax=ax1, label='bps')

im2 = ax2.imshow(daily_ask_dist.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax2.set_title('Ask Distance from Mid-Price (bps)', fontweight='bold')
ax2.set_ylabel('Depth Level')
ax2.set_yticks(range(10))
ax2.set_yticklabels([f'L{i}' for i in range(1, 11)])
ax2.set_xlabel('Day')
plt.colorbar(im2, ax=ax2, label='bps')

plt.tight_layout()
plt.show()

## 6. Multi-Instrument Comparison

Load all instruments and compare their average imbalance.

In [ ]:
import glob

files = sorted(glob.glob('*_5m_depth10_derived.csv.gz'))
print(f'Found {len(files)} instruments')

stats = []
for f in files:
    d = pd.read_csv(f)
    sym = d['instrument_symbol'].iloc[0]
    imb = (d['bid_volume_level_1'] - d['ask_volume_level_1']) / (d['bid_volume_level_1'] + d['ask_volume_level_1'])
    total_vol = d['interval_traded_volume'].sum()
    stats.append({
        'instrument': sym,
        'rows': len(d),
        'avg_imbalance_L1': imb.mean(),
        'std_imbalance_L1': imb.std(),
        'total_volume': total_vol,
        'avg_spread_L1_bps': (d['ask_distance_level_1'] + d['bid_distance_level_1']).mean(),
    })

comparison = pd.DataFrame(stats).set_index('instrument').sort_values('total_volume', ascending=False)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = [ACCENT if v > 0 else '#FF4444' for v in comparison['avg_imbalance_L1']]
ax.bar(comparison.index, comparison['avg_imbalance_L1'], color=colors, alpha=0.8)
ax.axhline(0, color='white', linewidth=0.5, alpha=0.3)
ax.set_ylabel('Average L1 Bid-Ask Imbalance')
ax.set_title('Cross-Instrument Imbalance Comparison', fontweight='bold')
ax.grid(alpha=0.15, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Feature Engineering Ideas

Here are some features you can derive from this dataset for ML models:

| Feature | Formula | Use Case |
|---|---|---|
| **Imbalance Ratio (L1-L10)** | `(bid_vol - ask_vol) / (bid_vol + ask_vol)` | Directional signal |
| **Depth-Weighted Imbalance** | Σ `imbalance_k × (1/k)` for k=1..10 | Weighted signal favoring top-of-book |
| **Total Depth** | Σ `bid_vol + ask_vol` for k=1..10 | Liquidity regime detection |
| **Depth Slope** | Linear regression slope of volume vs level | Wall detection |
| **Spread Momentum** | `diff(bid_distance_L1 + ask_distance_L1)` | Spread widening/tightening |
| **Volume-Distance Ratio** | `volume_Lk / distance_Lk` | Concentration score |
| **Cross-Level Divergence** | `imbalance_L1 - imbalance_L5` | Near vs far pressure gap |

---

🔗 **imbalancelabs.com** — Full dataset: 24 instruments, 12+ months, 47 columns/row.